In [1]:
import numpy as np
from scipy.optimize import minimize

# load params
mu = np.load("project3_outputs/mu_hat.npy")
Sigma = np.load("project3_outputs/Sigma_hat.npy")
gamma = 3.0
ones = np.array([1, 1, 1])
assets = ["SPY", "IEF", "GLD"]

print(f"mu    = {mu}")
print(f"gamma = {gamma}")
print(f"Sigma =")
for row in Sigma:
    print(f"        {row}")
print()

mu    = [ 0.00051575 -0.00024641  0.00032329]
gamma = 3.0
Sigma =
        [1.08500755e-04 5.42930040e-06 1.54645268e-05]
        [5.4293004e-06 2.6382581e-05 1.8884914e-05]
        [1.54645268e-05 1.88849140e-05 8.14528009e-05]



Unconstrained Optimal Portfolio
Closed form: w* = Σ⁻¹1/(1ᵀΣ⁻¹1) + (1/γ) Σ⁻¹(μ - (1ᵀΣ⁻¹μ)/(1ᵀΣ⁻¹1) · 1)

In [2]:
#Σ⁻¹
sigma_inv = np.linalg.inv(Sigma)

#Σ⁻¹𝟏
sigma_inv_ones = sigma_inv @ ones


GMVP = sigma_inv_ones / (ones @ sigma_inv_ones)

speculative = 1 / gamma * sigma_inv @ (mu - (ones @ sigma_inv @ mu) / (ones @ sigma_inv_ones) * ones)

w_star = GMVP + speculative

# stats
portfolio_mean = mu @ w_star
portfolio_var = w_star @ Sigma @ w_star
portfolio_utility = portfolio_mean - 0.5 * gamma * portfolio_var

print(f"GMVP component:        {np.round(GMVP, 6)}")
print(f"Speculative component: {np.round(speculative, 6)}")

for i, asset in enumerate(assets):
    print(f"w*_{asset} = {w_star[i]:+.6f}")

print(f"\nSum of weights:     {w_star.sum():.6f}")
print(f"Portfolio mean:     {portfolio_mean:.8f}")
print(f"Portfolio variance: {portfolio_var:.10f}")
print(f"Portfolio utility:  {portfolio_utility:.8f}")
print()

GMVP component:        [0.159458 0.773435 0.067107]
Speculative component: [ 1.726325 -4.004652  2.278327]
w*_SPY = +1.885783
w*_IEF = -3.231217
w*_GLD = +2.345434

Sum of weights:     1.000000
Portfolio mean:     0.00252705
Portfolio variance: 0.0008937694
Portfolio utility:  0.00118639



LONG-ONLY OPTIMAL PORTFOLIO: same objective, except all weights must be >= 0

In [3]:
# negative utility formula since scipy can only minimize. (Objective function)
def neg_utility(w):
    return -(mu @ w - 0.5 * gamma * (w @ Sigma @ w))

# weights sum must be == 1
constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]
bounds = [(0, None), (0, None), (0, None)]

best_util = -np.inf
best_w = None

for guess in [[1/3, 1/3, 1/3], [0.6, 0.1, 0.3], [0.2, 0.6, 0.2], [0.1, 0.2, 0.7], [1.0, 0.0, 0.0]]:
    res = minimize(neg_utility, x0=guess, method="SLSQP", bounds=bounds, constraints=constraints, options={"ftol": 1e-15, "maxiter": 1000})
    if res.success and -res.fun > best_util:
        # update best score
        best_util = -res.fun
        best_w = res.x

w_long = best_w

portfolio_mean_long = mu @ w_long
portfolio_var_long = w_long @ Sigma @ w_long
portfolio_utility_long = portfolio_mean_long - 0.5 * gamma * portfolio_var_long

for i, name in enumerate(assets):
    print(f"w*_{name} =            {w_long[i]:+.6f}")
print(f"\nSum of weights:     {w_long.sum():.6f}")
print(f"Portfolio mean:     {portfolio_mean_long:.8f}")
print(f"Portfolio variance: {portfolio_var_long:.10f}")
print(f"Portfolio utility:  {portfolio_utility_long:.8f}")

w*_SPY =            +0.818372
w*_IEF =            +0.000000
w*_GLD =            +0.181628

Sum of weights:     1.000000
Portfolio mean:     0.00048079
Portfolio variance: 0.0000799508
Portfolio utility:  0.00036087


In [4]:
print("Verification:")

# how much utility improves with a bit more weight
grad = mu - gamma * (Sigma @ w_long)

# marginal utility of active assets
l = np.mean(grad[w_long > 1e-8])  

for i, asset in enumerate(assets):
    if w_long[i] > 1e-8:
        nu_i = 0.0
        print(f"{asset}: w={w_long[i]:.6f} > 0, nu=0 (active, marginal util={grad[i]:.8f})")
    else:
        nu_i = l - grad[i]
        print(f"{asset}: w=0, nu={nu_i:.8f} > 0 (excluded)")

Verification:
SPY: w=0.818372 > 0, nu=0 (active, marginal util=0.00024094)
IEF: w=0, nu=0.00051097 > 0 (excluded)
GLD: w=0.181628 > 0, nu=0 (active, marginal util=0.00024094)


In [5]:
# Comparison between unconstrained and long-only
utility_loss = portfolio_utility - portfolio_utility_long
dist = np.linalg.norm(w_star - w_long)

print(f"Euclidean distance: {dist:.6f}")
print(f"Utility loss:       {utility_loss:.10f}\n")

for i, asset in enumerate(assets):
    print(f"{asset}:  unconstrained={w_star[i]:+.6f}   long only={w_long[i]:+.6f}")
print()
for i, asset in enumerate(assets):
    if w_star[i] < -1e-6:
        print(f"{asset} was shorted in the unconstrained solution but zeroed out by long-only")

Euclidean distance: 4.032640
Utility loss:       0.0008255263

SPY:  unconstrained=+1.885783   long only=+0.818372
IEF:  unconstrained=-3.231217   long only=+0.000000
GLD:  unconstrained=+2.345434   long only=+0.181628

IEF was shorted in the unconstrained solution but zeroed out by long-only


BOUNDED-WEIGHTS OPTIMAL PORTFOLIO: same objective, with lower and upper bounds on each weight

In [6]:
# Bounded-weights benchmark

import pandas as pd
from pathlib import Path

lower_bound = -0.5
upper_bound = 1.5


def neg_utility_bounded(w):
    return -(mu @ w - 0.5 * gamma * (w @ Sigma @ w))


constraints_bounded = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]
bounds_bounded = [(lower_bound, upper_bound), (lower_bound, upper_bound), (lower_bound, upper_bound)]

best_util_bounded = -np.inf
best_w_bounded = None
best_res_bounded = None

# Multiple feasible starting points for robustness
bounded_guesses = [
    [1 / 3, 1 / 3, 1 / 3],
    [0.6, 0.1, 0.3],
    [0.2, 0.6, 0.2],
    [0.1, 0.2, 0.7],
    [1.0, 0.0, 0.0],
    [1.2, -0.2, 0.0],
    [0.8, -0.5, 0.7],
    [-0.5, 1.0, 0.5],
]

for guess in bounded_guesses:
    guess = np.array(guess, dtype=float)

    res = minimize(
        neg_utility_bounded,
        x0=guess,
        method="SLSQP",
        bounds=bounds_bounded,
        constraints=constraints_bounded,
        options={"ftol": 1e-15, "maxiter": 1000},
    )

    if res.success and -res.fun > best_util_bounded:
        best_util_bounded = -res.fun
        best_w_bounded = res.x
        best_res_bounded = res

if best_w_bounded is None:
    raise RuntimeError("Bounded-weights optimization failed for all initial guesses.")

w_bounded = best_w_bounded

portfolio_mean_bounded = mu @ w_bounded
portfolio_var_bounded = w_bounded @ Sigma @ w_bounded
portfolio_utility_bounded = portfolio_mean_bounded - 0.5 * gamma * portfolio_var_bounded

print(f"Bounds: [{lower_bound}, {upper_bound}]")
for i, name in enumerate(assets):
    print(f"w*_{name} =            {w_bounded[i]:+.6f}")

print(f"\nSum of weights:     {w_bounded.sum():.6f}")
print(f"Portfolio mean:     {portfolio_mean_bounded:.8f}")
print(f"Portfolio variance: {portfolio_var_bounded:.10f}")
print(f"Portfolio utility:  {portfolio_utility_bounded:.8f}")
print(f"Solver success:     {best_res_bounded.success}")
print(f"Solver message:     {best_res_bounded.message}")

Bounds: [-0.5, 1.5]
w*_SPY =            +0.983544
w*_IEF =            -0.500000
w*_GLD =            +0.516456

Sum of weights:     1.000000
Portfolio mean:     0.00079743
Portfolio variance: 0.0001338979
Portfolio utility:  0.00059658
Solver success:     True
Solver message:     Optimization terminated successfully


In [7]:
print("Bounded-weight verification:")

grad_bounded = mu - gamma * (Sigma @ w_bounded)

# interior assets: strictly inside bounds
interior_mask = (w_bounded > lower_bound + 1e-8) & (w_bounded < upper_bound - 1e-8)

if np.any(interior_mask):
    lambda_hat_bounded = np.mean(grad_bounded[interior_mask])
else:
    lambda_hat_bounded = np.mean(grad_bounded)

for i, asset in enumerate(assets):
    if lower_bound + 1e-8 < w_bounded[i] < upper_bound - 1e-8:
        print(
            f"{asset}: interior, w={w_bounded[i]:+.6f}, "
            f"marginal util={grad_bounded[i]:.8f}, "
            f"lambda≈{lambda_hat_bounded:.8f}"
        )
    elif abs(w_bounded[i] - lower_bound) <= 1e-8:
        print(
            f"{asset}: at lower bound, w={w_bounded[i]:+.6f}, "
            f"marginal util={grad_bounded[i]:.8f}"
        )
    elif abs(w_bounded[i] - upper_bound) <= 1e-8:
        print(
            f"{asset}: at upper bound, w={w_bounded[i]:+.6f}, "
            f"marginal util={grad_bounded[i]:.8f}"
        )
    else:
        print(
            f"{asset}: near bound, w={w_bounded[i]:+.6f}, "
            f"marginal util={grad_bounded[i]:.8f}"
        )

Bounded-weight verification:
SPY: interior, w=+0.983544, marginal util=0.00017979, lambda≈0.00017979
IEF: at lower bound, w=-0.500000, marginal util=-0.00025212
GLD: interior, w=+0.516456, marginal util=0.00017979, lambda≈0.00017979


In [8]:
# Comparison across unconstrained, long-only, and bounded-weight solutions
utility_loss_bounded_vs_unconstrained = portfolio_utility - portfolio_utility_bounded
dist_unconstrained_bounded = np.linalg.norm(w_star - w_bounded)
dist_long_bounded = np.linalg.norm(w_long - w_bounded)

print("Comparison with bounded-weight benchmark:")
print(f"Distance to unconstrained benchmark: {dist_unconstrained_bounded:.6f}")
print(f"Distance to long-only benchmark:     {dist_long_bounded:.6f}")
print(f"Utility loss vs unconstrained:       {utility_loss_bounded_vs_unconstrained:.10f}\n")

for i, asset in enumerate(assets):
    print(
        f"{asset}:  "
        f"unconstrained={w_star[i]:+.6f}   "
        f"long only={w_long[i]:+.6f}   "
        f"bounded={w_bounded[i]:+.6f}"
    )

Comparison with bounded-weight benchmark:
Distance to unconstrained benchmark: 3.408628
Distance to long-only benchmark:     0.624013
Utility loss vs unconstrained:       0.0005898086

SPY:  unconstrained=+1.885783   long only=+0.818372   bounded=+0.983544
IEF:  unconstrained=-3.231217   long only=+0.000000   bounded=-0.500000
GLD:  unconstrained=+2.345434   long only=+0.181628   bounded=+0.516456


In [9]:
# Export bounded benchmark and trainable bounded action grid
# This is the part the training teammate can directly use.
# It creates:
# 1) bounded_benchmark_weights.csv  -> bounded optimum
# 2) bounded_feasible_weights.csv   -> discrete bounded action grid for Q-learning

outdir = Path("project3_outputs")
outdir.mkdir(exist_ok=True)

# Save bounded benchmark weights
bounded_benchmark_df = pd.DataFrame({
    "asset": assets,
    "weight": w_bounded,
})
bounded_benchmark_df.to_csv(outdir / "bounded_benchmark_weights.csv", index=False)

# Build bounded feasible action grid for downstream training
# step size can be adjusted by the training teammate later if needed
step = 0.1
values = np.round(np.arange(lower_bound, upper_bound + step / 2, step), 10)

feasible_actions = []
for w1 in values:
    for w2 in values:
        w3 = 1.0 - w1 - w2
        if lower_bound - 1e-10 <= w3 <= upper_bound + 1e-10:
            feasible_actions.append([round(w1, 10), round(w2, 10), round(w3, 10)])

bounded_feasible_df = pd.DataFrame(feasible_actions, columns=["w1", "w2", "w3"])
bounded_feasible_df = bounded_feasible_df.drop_duplicates().reset_index(drop=True)
bounded_feasible_df.to_csv(outdir / "bounded_feasible_weights.csv", index=False)

print("Saved bounded benchmark weights to:", outdir / "bounded_benchmark_weights.csv")
print("Saved bounded feasible action grid to:", outdir / "bounded_feasible_weights.csv")
print("Number of bounded feasible actions:", len(bounded_feasible_df))
print(bounded_feasible_df.head())

Saved bounded benchmark weights to: project3_outputs/bounded_benchmark_weights.csv
Saved bounded feasible action grid to: project3_outputs/bounded_feasible_weights.csv
Number of bounded feasible actions: 306
    w1   w2   w3
0 -0.5 -0.0  1.5
1 -0.5  0.1  1.4
2 -0.5  0.2  1.3
3 -0.5  0.3  1.2
4 -0.5  0.4  1.1


In [10]:
# Quick handoff preview for training the model
bounded_action_grid = pd.read_csv("project3_outputs/bounded_feasible_weights.csv")
print("Bounded action grid ready for training.")
print(bounded_action_grid.head())

Bounded action grid ready for training.
    w1   w2   w3
0 -0.5 -0.0  1.5
1 -0.5  0.1  1.4
2 -0.5  0.2  1.3
3 -0.5  0.3  1.2
4 -0.5  0.4  1.1
